# RAG 评测（DeepEval）：用可复用指标评估回答与检索

在做 RAG 时，我们常见的“好坏”其实可以拆成几类可衡量的指标：

- **Correctness（正确性）**：回答 vs 标准答案（需要 ground truth）
- **Faithfulness / Groundedness（忠实性/可溯源）**：回答是否被检索上下文支持（通常不需要 ground truth）
- **Contextual Relevancy（上下文相关性）**：检索上下文是否真的对当前问题有用、回答是否利用了上下文

这一节我们用 [DeepEval](https://github.com/confident-ai/deepeval) 演示如何把这些指标封装成可复用的评测，并对多个样本一次性跑评估。


## 0) 环境准备

本 notebook 不包含 `pip install`。请确保：

- 已设置 `OPENAI_API_KEY`（建议放在 `../.env`）
- 已安装 `deepeval`

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.insert(0, str(Path("..").resolve()))

from deepeval import evaluate
from deepeval.metrics import GEval, FaithfulnessMetric, ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

/tmp/ipykernel_983324/651054942.py:12: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [2]:
os.environ["HTTP_PROXY"] = os.getenv("HTTP_PROXY", "http://127.0.0.1:7890")
os.environ["HTTPS_PROXY"] = os.getenv("HTTPS_PROXY", "http://127.0.0.1:7890")
os.environ["ALL_PROXY"] = os.getenv("ALL_PROXY", "http://127.0.0.1:7890")

## 1) Correctness：回答是否与标准答案一致？

这里用 `GEval` 做一个“正确性”评估：给定 `expected_output`（标准答案）与 `actual_output`（模型输出），让评测模型判断是否“事实正确”。

注意：这类指标通常需要准备 ground truth。

In [3]:
correctness_metric = GEval(
    name="Correctness",
    # DeepEval 会用这个模型做 LLM-as-judge
    model="gpt-4.1-mini",
    evaluation_params=[
        LLMTestCaseParams.EXPECTED_OUTPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    evaluation_steps=[
        "Determine whether the actual output is factually correct based on the expected output."
    ],
)

gt_answer = "Madrid is the capital of Spain."
pred_answer = "MadriD."

test_case_correctness = LLMTestCase(
    input="What is the capital of Spain?",
    expected_output=gt_answer,
    actual_output=pred_answer,
)

correctness_metric.measure(test_case_correctness)
print("Correctness score:", correctness_metric.score)

Output()

Correctness score: 0.12051209706008763


## 2) Faithfulness / Groundedness：回答是否被检索上下文支持？

Faithfulness（忠实性）通常不要求你给 ground truth，而是检查：

- **回答是否可以从 `retrieval_context` 推导出来**
- 是否存在“上下文外的臆造/幻觉”


In [4]:
question = "what is 3+3?"
retrieval_context = ["6"]
generated_answer = "6"

faithfulness_metric = FaithfulnessMetric(
    threshold=0.7,
    model="gpt-4.1-mini",
    include_reason=False,
)

test_case_faith = LLMTestCase(
    input=question,
    actual_output=generated_answer,
    retrieval_context=retrieval_context,
)

faithfulness_metric.measure(test_case_faith)
print("Faithfulness score:", faithfulness_metric.score)

Output()

Faithfulness score: 1.0


## 3) Contextual Relevancy：上下文是否与问题相关、并对回答有帮助？

这里用一个例子演示：上下文里混入了无关句子，指标会评估上下文与问题/回答的相关性。

In [5]:
actual_output = "then go somewhere else."
retrieval_context = [
    "this is a test context",
    "mike is a cat",
    "if the shoes don't fit, then go somewhere else.",
]
gt_answer = "if the shoes don't fit, then go somewhere else."

relevance_metric = ContextualRelevancyMetric(
    threshold=1,
    model="gpt-4.1-mini",
    include_reason=True,
)

relevance_test_case = LLMTestCase(
    input="What if these shoes don't fit?",
    actual_output=actual_output,
    retrieval_context=retrieval_context,
    expected_output=gt_answer,
)

relevance_metric.measure(relevance_test_case)
print("Contextual relevancy score:", relevance_metric.score)
print("Reason:", relevance_metric.reason)

Output()

Contextual relevancy score: 0.3333333333333333
Reason: The score is 0.33 because while the retrieval context includes irrelevant information such as 'this is a test context' and 'mike is a cat,' it does contain a relevant statement: 'if the shoes don't fit, then go somewhere else.' This partial relevance justifies the low but non-zero score.


## 4) 多个样本 + 多个指标：一次性 evaluate

把多个 `LLMTestCase` 与多个指标一起传给 `evaluate(...)`，就能批量输出评估结果。

In [6]:
new_test_case = LLMTestCase(
    input="What is the capital of Spain?",
    expected_output="Madrid is the capital of Spain.",
    actual_output="MadriD.",
    retrieval_context=["Madrid is the capital of Spain."],
)

evaluate(
    test_cases=[relevance_test_case, new_test_case],
    metrics=[correctness_metric, faithfulness_metric, relevance_metric],
)

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              What if these shoes don't fit?                                                       │
│  │     Actual Output:      then go somewhere else.                                                              │
│  │     Expected Output:    if the shoes don't fit, then go somewhere else.                                      │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Correctness [GEval]  │ 0.31  │ 0.50      │ The actual output is incomplete and omits the         │
│              │                      │       │           │ conditional clause 'if the shoes don't fit,' which    │
│              │                      │       │           │ is essential for the meaning and correctness of the   │
│              │                      │       │           │ statement. While the latter part 'then go somewhere   │
│              │                      │       │           │ else.' is correct, the missing condition              │
│              │                      │       │           │ significantly reduces factual accuracy.               │
│        PASS  │ Faithfulness         │ 1.00  │ 0.70      │ N/A                                                   │
│        FAIL  │ Contextual Relevancy │ 0.33  │ 1.00      │ The score is 0.33 because most retrieval context      │
│              │                      │       │           │ statements like 'this is a test context' and 'mike    │
│              │                      │       │           │ is a cat' are irrelevant to the input about shoes     │
│              │                      │       │           │ not fitting, while only the statement 'if the shoes   │
│              │                      │       │           │ don't fit, then go somewhere else.' is relevant.      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              What is the capital of Spain?                                                        │
│  │     Actual Output:      MadriD.                                                                              │
│  │     Expected Output:    Madrid is the capital of Spain.                                                      │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇

⚠ WARNING: No hyperparameters logged.
» ]8;id=12278069;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 6.17s | token cost: 0.0031408 USD)
» Test Results (2 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=False, score=0.3064139587477135, reason="The actual output is incomplete and omits the conditional clause 'if the shoes don't fit,' which is essential for the meaning and correctness of the statement. While the latter part 'then go somewhere else.' is correct, the missing condition significantly reduces factual accuracy.", strict_mode=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost=0.00021080000000000003, input_tokens=263, output_tokens=66, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Determine whether the actual output is factually correct based on the expected output."\n] \n \nRubric:\nNone \n \nScore: 0.3064139587477135'), MetricData(name='Faithfulness', threshold=0.7, success=True, score=1.0, reason=None, strict_mode=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost=0.000588, i

## 5) 批量构造 LLMTestCase：把你的 RAG 日志变成评测数据

实际项目中，你通常会有四个并行列表：

- `questions`
- `gt_answers`（可选；有的话可以评 correctness）
- `generated_answers`
- `retrieved_documents`（每个问题对应一个 `list[str]` 的上下文）

下面这个小函数把它们打包成 `LLMTestCase[]`，就能直接喂给 `evaluate(...)`。

In [7]:
def create_deep_eval_test_cases(
    questions: list[str],
    gt_answers: list[str],
    generated_answers: list[str],
    retrieved_documents: list[list[str]],
) -> list[LLMTestCase]:
    return [
        LLMTestCase(
            input=question,
            expected_output=gt_answer,
            actual_output=generated_answer,
            retrieval_context=retrieved_document,
        )
        for question, gt_answer, generated_answer, retrieved_document in zip(
            questions,
            gt_answers,
            generated_answers,
            retrieved_documents,
        )
    ]